<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

### **Programación Orientada a Objetos (POO) — Parte 2**

En la parte anterior aprendimos a crear clases con atributos, métodos, `__str__`, type hints y docstrings. En esta segunda parte vamos a ver los mecanismos que hacen a la POO realmente poderosa:

- **Herencia** — una clase reutiliza y extiende otra
- **Sobreescritura de métodos** — el hijo redefine el comportamiento del padre
- **Polimorfismo** — objetos distintos responden al mismo mensaje de forma diferente
- **Encapsulamiento** — proteger los datos internos de un objeto
- **Métodos estáticos** (`@staticmethod`) — funciones utilitarias que viven dentro de la clase

Vamos a usar un **hilo conductor** a lo largo de toda la notebook: un sistema de empleados de una empresa.

---
### **Herencia**

La **herencia** permite que una clase (llamada **clase hija** o **subclase**) reutilice los atributos y métodos de otra clase (llamada **clase padre** o **superclase**), y además agregue o modifique comportamiento propio.

```python
class ClasePadre:
    ...

class ClaseHija(ClasePadre):   # ← hereda de ClasePadre
    ...
```

**¿Cuándo usarla?** Cuando dos o más clases comparten características comunes. En lugar de repetir el mismo código, lo ponemos en el padre y los hijos lo heredan.

| Concepto | Descripción |
|---|---|
| Clase padre | Contiene los atributos y métodos comunes |
| Clase hija | Hereda todo del padre y puede agregar o modificar |
| `super()` | Permite llamar al constructor o métodos del padre desde el hijo |

#### Clase padre: `Empleado`

In [ ]:
class Empleado:
    """Representa un empleado de la empresa."""

    def __init__(self, nombre: str, legajo: int, salario_base: float):
        """Inicializa el empleado.

        Args:
            nombre:       nombre completo
            legajo:       número de legajo único
            salario_base: salario mensual base
        """
        self.nombre: str        = nombre
        self.legajo: int        = legajo
        self.salario_base: float = salario_base

    def calcular_salario(self) -> float:
        """Devuelve el salario a cobrar. En la clase base es igual al salario base."""
        return self.salario_base

    def presentarse(self) -> str:
        """Devuelve una presentación del empleado."""
        return "Empleado: " + self.nombre + " | Legajo: " + str(self.legajo)

    def __str__(self) -> str:
        return (self.presentarse() +
                " | Salario: $" + str(self.calcular_salario()))


# Uso directo de la clase padre
emp = Empleado("Carlos Ruiz", 101, 3000.0)
print(emp)

#### Clase hija: `EmpleadoFullTime`

Un empleado full time tiene todo lo de `Empleado`, más un bono anual. Usamos `super().__init__()` para llamar al constructor del padre y no repetir código.

In [ ]:
class EmpleadoFullTime(Empleado):
    """Empleado con contrato de tiempo completo y bono anual."""

    def __init__(self, nombre: str, legajo: int, salario_base: float, bono_anual: float):
        """Inicializa el empleado full time.

        Args:
            nombre:       nombre completo
            legajo:       número de legajo
            salario_base: salario mensual base
            bono_anual:   bono que se paga una vez al año
        """
        super().__init__(nombre, legajo, salario_base)  # llama al __init__ del padre
        self.bono_anual: float = bono_anual             # atributo propio del hijo

    def __str__(self) -> str:
        return (self.presentarse() +
                " | Salario: $" + str(self.calcular_salario()) +
                " | Bono anual: $" + str(self.bono_anual) +
                " [Full Time]")


ft = EmpleadoFullTime("Ana García", 102, 5000.0, 10000.0)
print(ft)

# Hereda los atributos del padre
print("Nombre:",  ft.nombre)
print("Legajo:",  ft.legajo)
print("Bono:",    ft.bono_anual)

# Hereda los métodos del padre
print(ft.presentarse())

#### Clase hija: `EmpleadoPartTime`

In [ ]:
class EmpleadoPartTime(Empleado):
    """Empleado que trabaja por horas."""

    def __init__(self, nombre: str, legajo: int, valor_hora: float, horas_mes: int):
        """Inicializa el empleado part time.

        Args:
            nombre:     nombre completo
            legajo:     número de legajo
            valor_hora: pago por hora trabajada
            horas_mes:  horas trabajadas en el mes
        """
        salario_calculado: float = valor_hora * horas_mes
        super().__init__(nombre, legajo, salario_calculado)
        self.valor_hora: float = valor_hora
        self.horas_mes: int    = horas_mes

    def __str__(self) -> str:
        return (self.presentarse() +
                " | $" + str(self.valor_hora) + "/h x " + str(self.horas_mes) + "h" +
                " | Salario: $" + str(self.calcular_salario()) +
                " [Part Time]")


pt = EmpleadoPartTime("Bruno López", 103, 50.0, 80)
print(pt)

#### Verificar la jerarquía de clases con `isinstance()`

`isinstance(objeto, Clase)` devuelve `True` si el objeto es instancia de esa clase **o de cualquiera de sus padres**.

In [ ]:
ft = EmpleadoFullTime("Ana", 102, 5000.0, 10000.0)
pt = EmpleadoPartTime("Bruno", 103, 50.0, 80)

print(isinstance(ft, EmpleadoFullTime))  # True
print(isinstance(ft, Empleado))          # True  ← también es Empleado
print(isinstance(pt, EmpleadoPartTime))  # True
print(isinstance(pt, EmpleadoFullTime))  # False
print(isinstance(pt, Empleado))          # True

#### Herencia múltiple

Python permite que una clase herede de **más de una clase padre** al mismo tiempo:

```python
class ClaseHija(PadreA, PadreB):
    ...
```

Es una característica avanzada que puede generar conflictos si los padres tienen métodos con el mismo nombre. Para la mayoría de los casos del día a día, la herencia simple es suficiente y más clara.

---
### **Sobreescritura de métodos (Method Override)**

Cuando una clase hija **redefine un método del padre**, decimos que lo está **sobreescribiendo**. El hijo usa el mismo nombre de método pero con un comportamiento diferente.

Esto es muy útil cuando la lógica general vive en el padre, pero cada tipo de objeto necesita una variante específica de esa lógica.

In [ ]:
class Empleado:
    """Clase base de empleados."""

    def __init__(self, nombre: str, legajo: int, salario_base: float):
        self.nombre: str         = nombre
        self.legajo: int         = legajo
        self.salario_base: float = salario_base

    def calcular_salario(self) -> float:
        """Salario base sin adicionales."""
        return self.salario_base

    def presentarse(self) -> str:
        return "Empleado: " + self.nombre + " | Legajo: " + str(self.legajo)

    def __str__(self) -> str:
        return self.presentarse() + " | Salario: $" + str(self.calcular_salario())


class EmpleadoFullTime(Empleado):
    """Empleado full time con bono y adicional por antigüedad."""

    def __init__(self, nombre: str, legajo: int, salario_base: float,
                 bono_anual: float, anios_antiguedad: int = 0):
        super().__init__(nombre, legajo, salario_base)
        self.bono_anual: float      = bono_anual
        self.anios_antiguedad: int  = anios_antiguedad

    def calcular_salario(self) -> float:
        """Sobreescribe el método del padre: agrega 2% por año de antigüedad."""
        adicional_antiguedad: float = self.salario_base * 0.02 * self.anios_antiguedad
        return self.salario_base + adicional_antiguedad

    def __str__(self) -> str:
        return (self.presentarse() +
                " | Salario: $" + str(self.calcular_salario()) +
                " (" + str(self.anios_antiguedad) + " años antigüedad) [Full Time]")


class EmpleadoPartTime(Empleado):
    """Empleado part time: cobra por horas."""

    def __init__(self, nombre: str, legajo: int, valor_hora: float, horas_mes: int):
        super().__init__(nombre, legajo, valor_hora * horas_mes)
        self.valor_hora: float = valor_hora
        self.horas_mes: int    = horas_mes

    def calcular_salario(self) -> float:
        """Sobreescribe el método del padre: calcula según horas trabajadas."""
        return self.valor_hora * self.horas_mes

    def __str__(self) -> str:
        return (self.presentarse() +
                " | $" + str(self.valor_hora) + "/h x " + str(self.horas_mes) + "h" +
                " | Salario: $" + str(self.calcular_salario()) + " [Part Time]")


# Cada clase calcula el salario a su manera
emp_base = Empleado("Carlos Ruiz", 101, 3000.0)
emp_ft   = EmpleadoFullTime("Ana García", 102, 5000.0, 10000.0, anios_antiguedad=5)
emp_pt   = EmpleadoPartTime("Bruno López", 103, 50.0, 80)

print(emp_base)
print(emp_ft)
print(emp_pt)

---
### **Polimorfismo**

**Polimorfismo** significa "muchas formas". En POO, es la capacidad de tratar objetos de diferentes clases de forma **uniforme**, llamando al mismo método y obteniendo el comportamiento correcto para cada tipo.

Es la consecuencia natural de la herencia y la sobreescritura: si todos los empleados tienen `calcular_salario()`, podemos llamar a ese método **sin preocuparnos por el tipo específico** de cada uno.

> En la práctica: podemos poner objetos de distintas clases hijas en una misma lista y procesarlos con el mismo `for`.

In [ ]:
# Lista con empleados de distintos tipos
empleados: list[Empleado] = [
    Empleado("Carlos Ruiz",    101, 3000.0),
    EmpleadoFullTime("Ana García",   102, 5000.0, 10000.0, anios_antiguedad=5),
    EmpleadoPartTime("Bruno López",  103, 50.0, 80),
    EmpleadoFullTime("Carla Pérez",  104, 4500.0, 8000.0,  anios_antiguedad=2),
    EmpleadoPartTime("Diego Martín", 105, 60.0, 60),
]

# Polimorfismo: el mismo método, comportamiento distinto según el tipo
print("=== LIQUIDACIÓN DE SUELDOS ===")
for emp in empleados:
    # calcular_salario() ejecuta la versión correcta para cada tipo
    print(emp.nombre, "→ $", emp.calcular_salario())

In [ ]:
# Calcular el total de la nómina usando polimorfismo
total_nomina: float = 0.0

for emp in empleados:
    total_nomina = total_nomina + emp.calcular_salario()

print("Total nómina del mes: $", total_nomina)

In [ ]:
# Separar por tipo usando isinstance()
full_times: list[EmpleadoFullTime]  = []
part_times: list[EmpleadoPartTime]  = []

for emp in empleados:
    if isinstance(emp, EmpleadoFullTime):
        full_times.append(emp)
    elif isinstance(emp, EmpleadoPartTime):
        part_times.append(emp)

print("Full time:", len(full_times))
for ft in full_times:
    print(" ", ft)

print("Part time:", len(part_times))
for pt in part_times:
    print(" ", pt)

---
### **Encapsulamiento**

El **encapsulamiento** consiste en **proteger los datos internos** de un objeto, evitando que se modifiquen directamente desde afuera de la clase.

**¿Por qué proteger atributos?** Porque algunos datos no deberían cambiarse libremente. Por ejemplo, el saldo de una cuenta bancaria no debería poder modificarse así:
```python
cuenta.saldo = -99999   # ← esto no debería ser posible
```

En Python los atributos privados se marcan con **doble guión bajo** `__` al inicio del nombre. Python los "oculta" y dificulta el acceso directo desde afuera.

Para **leer** o **modificar** atributos privados de forma controlada usamos métodos especiales:
- `get_atributo()` → leer el valor
- `set_atributo(valor)` → modificar el valor con validación

In [ ]:
# Sin encapsulamiento: cualquiera puede romper el saldo
class CuentaSinProteccion:
    def __init__(self, titular: str, saldo: float):
        self.titular: str  = titular
        self.saldo: float  = saldo     # atributo público


cuenta = CuentaSinProteccion("Ana", 1000.0)
print("Saldo:", cuenta.saldo)

# Nada impide esto:
cuenta.saldo = -99999.0
print("Saldo adulterado:", cuenta.saldo)

In [ ]:
# Con encapsulamiento: el saldo está protegido
class CuentaBancaria:
    """Cuenta bancaria con saldo encapsulado."""

    def __init__(self, titular: str, saldo_inicial: float = 0.0):
        """Inicializa la cuenta."""
        self.titular: str    = titular
        self.__saldo: float  = saldo_inicial   # ← atributo PRIVADO (doble __)

    # Getter: método para LEER el saldo
    def get_saldo(self) -> float:
        """Devuelve el saldo actual."""
        return self.__saldo

    def depositar(self, monto: float) -> None:
        """Suma monto al saldo. El monto debe ser positivo."""
        if monto <= 0:
            print("Error: el monto debe ser positivo")
            return
        self.__saldo = self.__saldo + monto
        print("Depósito de $", monto, "→ Saldo: $", self.__saldo)

    def retirar(self, monto: float) -> bool:
        """Retira monto si hay saldo suficiente."""
        if monto <= 0:
            print("Error: el monto debe ser positivo")
            return False
        if monto > self.__saldo:
            print("Saldo insuficiente. Saldo actual: $", self.__saldo)
            return False
        self.__saldo = self.__saldo - monto
        print("Retiro de $", monto, "→ Saldo: $", self.__saldo)
        return True

    def __str__(self) -> str:
        return "Cuenta de " + self.titular + " | Saldo: $" + str(self.__saldo)


cuenta = CuentaBancaria("Ana García", 1000.0)
print(cuenta)

cuenta.depositar(500.0)
cuenta.retirar(200.0)
cuenta.retirar(5000.0)   # falla con mensaje
cuenta.depositar(-100.0) # falla con mensaje

print("Saldo final: $", cuenta.get_saldo())

In [ ]:
# Intentar acceder al atributo privado directamente genera un error
cuenta = CuentaBancaria("Ana", 1000.0)

# Esto genera AttributeError:
# print(cuenta.__saldo)

# La forma correcta es usando el getter:
print("Saldo:", cuenta.get_saldo())

#### Setter: modificar con validación

In [ ]:
class Empleado:
    """Empleado con salario encapsulado."""

    def __init__(self, nombre: str, salario: float):
        self.nombre: str      = nombre
        self.__salario: float = salario   # privado

    def get_salario(self) -> float:
        """Devuelve el salario."""
        return self.__salario

    def set_salario(self, nuevo_salario: float) -> None:
        """Actualiza el salario con validación: debe ser mayor a cero."""
        if nuevo_salario <= 0:
            print("Error: el salario debe ser mayor a cero")
            return
        self.__salario = nuevo_salario
        print("Salario actualizado a $", self.__salario)

    def __str__(self) -> str:
        return self.nombre + " | Salario: $" + str(self.__salario)


emp = Empleado("Bruno", 3000.0)
print(emp)

emp.set_salario(3500.0)    # válido
emp.set_salario(-500.0)    # inválido, no se aplica
print(emp)

---
### **Métodos estáticos (`@staticmethod`)**

Un **método estático** es una función que vive dentro de una clase pero **no recibe `self`**: no accede ni modifica los atributos de ningún objeto.

Se usa cuando la funcionalidad está **lógicamente relacionada con la clase** pero no depende de ningún objeto en particular. Son funciones utilitarias agrupadas bajo el nombre de la clase.

```python
class MiClase:
    @staticmethod
    def mi_metodo_estatico(param: tipo) -> tipo:
        ...
```

Se llaman usando el nombre de la clase (sin crear un objeto):
```python
MiClase.mi_metodo_estatico(argumento)
```

In [ ]:
class Empleado:
    """Empleado con métodos estáticos utilitarios."""

    def __init__(self, nombre: str, legajo: int, salario_base: float):
        self.nombre: str          = nombre
        self.legajo: int          = legajo
        self.salario_base: float  = salario_base

    def calcular_salario(self) -> float:
        return self.salario_base

    # Método estático: no usa self, es una utilidad de la clase
    @staticmethod
    def calcular_salario_neto(salario_bruto: float, tasa_impuesto: float = 0.17) -> float:
        """Calcula el salario neto descontando impuestos.

        Args:
            salario_bruto: monto antes de impuestos
            tasa_impuesto: porcentaje de descuento (por defecto 17%)

        Returns:
            Salario neto después de impuestos.
        """
        return salario_bruto * (1 - tasa_impuesto)

    @staticmethod
    def es_legajo_valido(legajo: int) -> bool:
        """Verifica que el legajo tenga entre 3 y 6 dígitos."""
        return 100 <= legajo <= 999999

    def __str__(self) -> str:
        return self.nombre + " | Legajo: " + str(self.legajo) + " | Salario: $" + str(self.salario_base)


# Llamada SIN crear un objeto (usando el nombre de la clase)
neto: float = Empleado.calcular_salario_neto(5000.0)
print("Salario neto (17% impuesto): $", neto)

neto_exec: float = Empleado.calcular_salario_neto(10000.0, tasa_impuesto=0.25)
print("Salario neto (25% impuesto): $", neto_exec)

print(Empleado.es_legajo_valido(101))     # True
print(Empleado.es_legajo_valido(9999999)) # False

In [ ]:
# También se pueden llamar desde un objeto, aunque no es lo más común
emp = Empleado("Ana", 102, 5000.0)

neto: float = emp.calcular_salario_neto(emp.calcular_salario())
print("Salario bruto: $", emp.calcular_salario())
print("Salario neto:  $", neto)

#### Otro ejemplo: métodos estáticos en `CuentaBancaria`

In [ ]:
class CuentaBancaria:
    """Cuenta bancaria con utilidades estáticas."""

    def __init__(self, titular: str, saldo_inicial: float = 0.0):
        self.titular: str   = titular
        self.__saldo: float = saldo_inicial

    def get_saldo(self) -> float:
        return self.__saldo

    def depositar(self, monto: float) -> None:
        if CuentaBancaria.es_monto_valido(monto):   # usa el método estático
            self.__saldo = self.__saldo + monto

    @staticmethod
    def es_monto_valido(monto: float) -> bool:
        """Verifica que un monto sea positivo y no supere el límite de transacción."""
        return 0 < monto <= 100000

    @staticmethod
    def calcular_interes(saldo: float, tasa_anual: float, meses: int) -> float:
        """Calcula el interés simple sobre un saldo.

        Args:
            saldo:      capital inicial
            tasa_anual: tasa de interés anual (ej: 0.05 = 5%)
            meses:      cantidad de meses

        Returns:
            Monto de interés generado.
        """
        return saldo * tasa_anual * (meses / 12)


# Usar los métodos estáticos sin crear cuenta
print(CuentaBancaria.es_monto_valido(500.0))     # True
print(CuentaBancaria.es_monto_valido(-100.0))    # False
print(CuentaBancaria.es_monto_valido(200000.0))  # False

interes: float = CuentaBancaria.calcular_interes(10000.0, 0.05, 6)
print("Interés en 6 meses al 5% anual: $", interes)

---
### **Ejemplo integrador: sistema de empleados completo**

Combinamos todo lo visto: herencia, sobreescritura, polimorfismo, encapsulamiento y métodos estáticos.

In [ ]:
class Empleado:
    """Clase base del sistema de empleados."""

    def __init__(self, nombre: str, legajo: int, salario_base: float):
        if not Empleado.es_legajo_valido(legajo):
            print("Advertencia: legajo fuera de rango")
        self.nombre: str          = nombre
        self.legajo: int          = legajo
        self.__salario_base: float = salario_base   # encapsulado

    def get_salario_base(self) -> float:
        return self.__salario_base

    def set_salario_base(self, nuevo: float) -> None:
        if nuevo <= 0:
            print("Error: salario debe ser positivo")
            return
        self.__salario_base = nuevo

    def calcular_salario(self) -> float:
        """Devuelve el salario bruto. Cada subclase puede sobreescribirlo."""
        return self.__salario_base

    def calcular_salario_neto(self) -> float:
        """Devuelve el salario neto usando el método estático."""
        return Empleado.aplicar_impuesto(self.calcular_salario())

    @staticmethod
    def aplicar_impuesto(salario_bruto: float, tasa: float = 0.17) -> float:
        """Calcula el salario neto descontando impuestos."""
        return round(salario_bruto * (1 - tasa), 2)

    @staticmethod
    def es_legajo_valido(legajo: int) -> bool:
        """Verifica que el legajo sea un número de 3 a 6 dígitos."""
        return 100 <= legajo <= 999999

    def __str__(self) -> str:
        return ("[" + str(self.legajo) + "] " + self.nombre +
                " | Bruto: $" + str(self.calcular_salario()) +
                " | Neto: $"  + str(self.calcular_salario_neto()))


class EmpleadoFullTime(Empleado):
    """Empleado de planta con adicional por antigüedad."""

    def __init__(self, nombre: str, legajo: int, salario_base: float, anios_antiguedad: int = 0):
        super().__init__(nombre, legajo, salario_base)
        self.anios_antiguedad: int = anios_antiguedad

    def calcular_salario(self) -> float:
        """Salario base + 2% por año de antigüedad."""
        adicional: float = self.get_salario_base() * 0.02 * self.anios_antiguedad
        return round(self.get_salario_base() + adicional, 2)

    def __str__(self) -> str:
        return super().__str__() + " [FT | " + str(self.anios_antiguedad) + " años]"


class EmpleadoPartTime(Empleado):
    """Empleado por horas."""

    def __init__(self, nombre: str, legajo: int, valor_hora: float, horas_mes: int):
        super().__init__(nombre, legajo, valor_hora * horas_mes)
        self.valor_hora: float = valor_hora
        self.horas_mes: int    = horas_mes

    def calcular_salario(self) -> float:
        """Salario según horas trabajadas en el mes."""
        return round(self.valor_hora * self.horas_mes, 2)

    def __str__(self) -> str:
        return super().__str__() + " [PT | " + str(self.horas_mes) + "h]"


# ── Programa principal ───────────────────────────────────────────────────
empresa: list[Empleado] = [
    EmpleadoFullTime("Ana García",    101, 5000.0, anios_antiguedad=8),
    EmpleadoFullTime("Carla Pérez",   102, 4500.0, anios_antiguedad=2),
    EmpleadoPartTime("Bruno López",   103, 50.0,   80),
    EmpleadoPartTime("Diego Martín",  104, 60.0,   60),
    Empleado("Elena Ruiz",            105, 3000.0),
]

print("=" * 60)
print("LIQUIDACIÓN MENSUAL")
print("=" * 60)
for emp in empresa:
    print(emp)

# Polimorfismo: calcular_salario() de cada tipo
total_bruto: float = 0.0
total_neto: float  = 0.0
for emp in empresa:
    total_bruto = total_bruto + emp.calcular_salario()
    total_neto  = total_neto  + emp.calcular_salario_neto()

print("=" * 60)
print("Total bruto:  $", round(total_bruto, 2))
print("Total neto:   $", round(total_neto,  2))

---
### **Resumen**

| Concepto | Sintaxis clave | ¿Para qué sirve? |
|---|---|---|
| Herencia | `class Hijo(Padre):` | Reutilizar código del padre |
| Constructor heredado | `super().__init__(...)` | Inicializar la parte del padre |
| Override | Redefinir método en el hijo | Cambiar el comportamiento heredado |
| Polimorfismo | Mismo método, distinto resultado | Tratar objetos distintos de forma uniforme |
| Verificar tipo | `isinstance(obj, Clase)` | Saber de qué tipo es un objeto |
| Atributo privado | `self.__atributo` | Proteger datos internos |
| Getter | `def get_x(self) -> tipo:` | Leer un atributo privado |
| Setter | `def set_x(self, v: tipo):` | Modificar con validación |
| Método estático | `@staticmethod` | Utilidad sin acceso a `self` |
| Llamar estático | `Clase.metodo(args)` | Sin crear objeto |
| Herencia múltiple | `class Hijo(A, B):` | Existe pero usar con cuidado |